# 🏗️ Model Building & Comparison
## Taxi Fare Prediction

This notebook builds and compares multiple regression models:
- **Linear Regression**: Baseline model
- **Ridge Regression**: L2 regularization
- **Lasso Regression**: L1 regularization & feature selection
- **ElasticNet**: Combined L1 & L2 regularization
- **Polynomial Regression**: Non-linear relationships
- **Gradient Boosting**: Advanced ensemble method

**Evaluation Metrics**: MAE, RMSE, R², MSE, Cross-Validation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## Section 1: Load and Prepare Data

In [ ]:
# Load transformed dataset
df = pd.read_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\taxi_fare_transformed.csv')

print("=" * 80)
print("MODEL BUILDING & COMPARISON")
print("=" * 80)

# Separate features and target
target = 'fare_amount' if 'fare_amount' in df.columns else df.columns[-1]
X = df.drop(columns=[target]) if target in df.columns else df.iloc[:, :-1]
y = df[target] if target in df.columns else df.iloc[:, -1]

print(f"\nDataset shape: {X.shape}")
print(f"Train-test split (80-20)...\n")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")

## Section 2: Build and Train Multiple Models

In [ ]:
# Define models to train
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Lasso Regression': Lasso(alpha=0.01, random_state=42),
    'ElasticNet Regression': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

print("=" * 80)
print("TRAINING MODELS")
print("=" * 80)

trained_models = {}
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[model_name] = model
    print(f"  ✓ {model_name} trained successfully")

print("\n✓ All models trained!")

## Section 3: Model Evaluation

In [ ]:
# Evaluate all models
print("=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

results = []
for model_name, model in trained_models.items():
    # Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Metrics
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)
    
    mae_test = mean_absolute_error(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)
    
    results.append({
        'Model': model_name,
        'Train_R²': r2_train,
        'Test_R²': r2_test,
        'Train_RMSE': rmse_train,
        'Test_RMSE': rmse_test,
        'Train_MAE': mae_train,
        'Test_MAE': mae_test
    })

# Create results dataframe
results_df = pd.DataFrame(results)
print("\n" + results_df.to_string(index=False))

# Save results
results_df.to_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\model_comparison_results.csv', index=True)
print("\n✓ Model comparison results saved!")

## Section 4: Cross-Validation

In [ ]:
# Cross-validation
print("\n" + "=" * 80)
print("CROSS-VALIDATION (5-Fold)")
print("=" * 80)

cv_results = {}
for model_name, model in trained_models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    cv_results[model_name] = {
        'Mean_CV_R²': cv_scores.mean(),
        'Std_CV_R²': cv_scores.std(),
        'CV_Scores': cv_scores
    }
    print(f"\n{model_name}:")
    print(f"  Mean R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Fold scores: {[f'{s:.4f}' for s in cv_scores]}")

print("\n✓ Cross-validation completed!")

## Section 5: Visualize Comparison

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

# R² Score Comparison
results_df.set_index('Model')[['Train_R²', 'Test_R²']].plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('R² Score Comparison')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim([0, 1])
axes[0].legend(['Train', 'Test'])

# RMSE Comparison
results_df.set_index('Model')[['Train_RMSE', 'Test_RMSE']].plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('RMSE Comparison')
axes[1].set_ylabel('RMSE ($)')
axes[1].legend(['Train', 'Test'])

# MAE Comparison
results_df.set_index('Model')[['Train_MAE', 'Test_MAE']].plot(kind='bar', ax=axes[2], color=['steelblue', 'coral'])
axes[2].set_title('MAE Comparison')
axes[2].set_ylabel('MAE ($)')
axes[2].legend(['Train', 'Test'])

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("✓ Visualizations completed!")